# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

This notebook turns the Week 4-6 work into a content action playbook: a ranked review queue with reason codes, a rule flag as the reason and one of four actions per row, the decay and refresh finding, intended use, limits, human-review rules, a no-go list, cost and value framing, and monitoring triggers. It exports the queue and two figures for next week's paper.

**Nothing new is decided about the data or the models here.** The frame builder is Week 6's `build_frame`, the models are Week 5's `models` dict, the metric functions are Week 4's. Every number quoted in prose comes from a committed receipt (`baseline_metrics.json`, `model_metrics.json`, `validation_audit_metrics.json`) or from a cell in this notebook.

**What Week 6 measured, and what it means for the playbook:**

- Client-grouped CV inside March: rule P@50 0.512, LR 0.636, RF 0.608, against a 0.51 fold base rate. Both models beat the rule inside the month they were fit.
- One month forward (fit on March, score April): LR 0.54, RF 0.14, April base rate 0.56. Neither model beat the base rate at the head of the queue. RF collapsed; LR matched a coin flip.
- Walk-forward on the fixed 23-client panel: LR 0.56-0.70, RF 0.28-0.48, rule 0.38. More history did not fix it.
- The rule's top band is 24,462 pages tied at score 1.00, with a 50.7% decline rate. The model's only job is to order that band.
- The three wrong cases in Week 5 and Week 6 were all the same pattern: position 2 or better, CTR under 0.2%, 1.6k-17k impressions, clustered in one client. A page-level refresh cannot fix that.

So the queue in this notebook is built the way Week 6 validated it: fit on the March frame, score the April frame. LR orders it because it beat the rule out of month and RF did not, and the playbook is framed as a review order, not a forecast.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The rule explains, the model orders.** Each row in the queue carries three things: the page identifier, the model probability that sets the order, and the audited Week-4 rule flag that supplies the reason. The rule flags are the five reason codes from `w04_baseline_score.ipynb`, re-implemented verbatim: `stale_low_ctr_top10`, `stale`, `low_ctr_top10`, `visible`, `low_visibility`. The Week-4 signal audit confirmed staleness and CTR-vs-position as directional signals, so an editor can trust the flag as a reason. The model score is not explained to the editor; it only decides who goes first.

**How the order is built.**

1. Primary key: LR probability of decline, from a model fit on the March frame and applied to the April frame. This is the Week-6 Row-3 design, so the head of the queue can be read against a receipt (LR P@50 0.54, April base 0.56) instead of an in-sample number.
2. Tie-break: April impressions, descending. Between two pages the model cannot separate, the one with more traffic at stake goes first. The same tie-break is used everywhere in this notebook, including the precision checks.
3. Rows removed by the Section-3 gates never enter the budget queue.

**Budget.** The review budget is 50 pages a month, which is why the success metric since Week 2 has been precision@50. The exported queue is the top 50 eligible rows.

**Four actions.** Every queued row gets one of four actions, decided from the rule flag, the model score, and the March-to-April impressions ratio:

- `verify_then_review`: April impressions moved more than 7x against March in either direction. Check for a tracking or analytics change before an editor touches it.
- `monitor_only`: April impressions are at least 1.5x March. The page is rising; do not modify it. Durability over more than one month cannot be measured from two frames, so this is a one-month riser test and is disclosed as such.
- `investigate_quiet_risk`: model score >= 0.70 but the rule flag is `visible` or `low_visibility` (no risk flag). The model sees something the rule does not, and it cannot explain itself, so a person checks it.
- `review_before_revert`: everything else. Inspect the page, check search intent, form a hypothesis, then change content.

**Refresh hint.** For rows that reach an editor, a second column says what kind of work the features point at, in Week-4 flag terms: `stale_low_ctr_top10` and `low_ctr_top10` point at title, meta and snippet; `stale` points at a freshness pass; a flag with position <= 2 and CTR < 0.2% points at a query-mix diagnosis, not a rewrite, because that was the wrong-case pattern in Week 5 and Week 6.

**What P@50 means here.** Inside March, an LR-ranked top-50 held about 0.64 declining pages against a 0.51 fold base rate and a 0.507 tie-band rate: roughly 13 more true declines per 100 reviewed than random tie-breaking, in the month the model was fit. One month forward the measured lift over the base rate was gone (0.54 vs 0.56), although LR stayed above the rule (0.44). The queue precision printed below is the out-of-time number for the April frame, checked against the Week-6 receipt. It is not a new result.

In [ ]:
# Setup: Week-6 frame builder, Week-5 models, Week-4 metric functions, and the receipts.
%pip -q install duckdb huggingface_hub scikit-learn matplotlib

import os
import getpass
import json
import subprocess
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SEED = 42  # every stochastic step in this notebook uses this one seed

print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| duckdb", duckdb.__version__)

# --- repo root: ../../ when run from work/notebooks; clone into /content on bare Colab ---
REPO = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "work" / "outputs").is_dir() and (p / "skills").is_dir():
        REPO = p
        break
if REPO is None:
    REPO = Path("/content/FlyRank-ML-Internship")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git", str(REPO)], check=True)
OUT_DIR = REPO / "work" / "outputs"
FIG_DIR = REPO / "work" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

# --- receipts ---
w4 = json.loads((OUT_DIR / "baseline_metrics.json").read_text())
w5 = json.loads((OUT_DIR / "model_metrics.json").read_text())
w6 = json.loads((OUT_DIR / "validation_audit_metrics.json").read_text())
GCV = w6["grouped_cv"]["summary"]
TF = w6["time_forward"]
TIE = w6["rule_tie_band"]
print(f"\nW5/W6 grouped-CV P@50 means: rule {GCV['baseline']['p50_mean']:.3f} | LR {GCV['logistic_regression']['p50_mean']:.3f} | RF {GCV['random_forest']['p50_mean']:.3f}")
print(f"W6 time-forward P@50:        rule {TF['metrics']['baseline']['p50']:.2f} | LR {TF['metrics']['logistic_regression']['p50']:.2f} | RF {TF['metrics']['random_forest']['p50']:.2f}  (April base {TF['test_base_rate']:.3f})")
print(f"W6 rule tie band (March):    n={TIE['n']:,}  decline rate {TIE['decline_rate']:.3f}")

# --- token resolution: env var -> Colab secret -> repo .env -> interactive prompt ---
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN and (REPO / ".env").exists():
    for line in (REPO / ".env").read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            HF_TOKEN = line.split("=", 1)[1].strip()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")

FEATURES = [
    "log_recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "recent30_active_days",
    "content_age_days",
]
TARGET = "is_declining_next30"


def _months_between(start, end):
    months = []
    y, m = start.year, start.month
    while (y, m) <= (end.year, end.month):
        months.append(f"{y:04d}-{m:02d}")
        m += 1
        if m == 13:
            y, m = y + 1, 1
    return months


def build_frame(feature_month: str):
    # Week-6 builder, unchanged: Week-5 SQL parameterized by feature month.
    fact_feat = f"read_parquet('{REL}/fact_content_daily_performance/{feature_month}/*.parquet')"
    cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {fact_feat}").fetchone()[0]
    recent_start = cutoff_date - timedelta(days=29)
    label_start = cutoff_date + timedelta(days=1)
    label_end = cutoff_date + timedelta(days=30)
    fut_months = _months_between(label_start, label_end)
    fut_paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in fut_months)
    fact_fut = f"read_parquet([{fut_paths}])"
    part_min, part_max = con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM {fact_fut}").fetchone()
    print(f"Feature month {feature_month} | cutoff {cutoff_date} | label window {label_start} .. {label_end} | "
          f"label partitions {fut_months} | bounds {part_min} .. {part_max}")
    assert str(part_min) == str(label_start), "Label partitions start late - label window uncovered"
    assert part_max >= label_end, "Label partitions end before the label window - label window uncovered"

    frame = con.sql(f'''
        WITH recent AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) AS recent30_impressions,
                   SUM(gsc_clicks) AS recent30_clicks,
                   AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
                   COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
                   COUNT(DISTINCT report_date) AS recent30_days
            FROM {fact_feat}
            WHERE report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
              AND gsc_data_available IS TRUE
            GROUP BY 1, 2
        ),
        future AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) AS future30_impressions,
                   COUNT(DISTINCT report_date) AS future30_days
            FROM {fact_fut}
            WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
              AND gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT r.client_hash_id, r.content_hash_id,
               r.recent30_impressions,
               LN(1 + r.recent30_impressions) AS log_recent30_impressions,
               100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
               r.recent30_avg_position,
               r.recent30_active_days,
               DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
               f.future30_impressions,
               f.future30_days,
               CASE WHEN r.recent30_impressions >= 100
                     AND f.future30_impressions < 0.80 * r.recent30_impressions THEN 1 ELSE 0 END AS is_declining_next30
        FROM recent r
        INNER JOIN future f USING (client_hash_id, content_hash_id)
        LEFT JOIN (SELECT client_hash_id, content_hash_id, content_created_date FROM {DIM_CONTENT}) c
               USING (client_hash_id, content_hash_id)
        WHERE r.recent30_days >= 14 AND f.future30_days >= 14 AND r.recent30_impressions >= 100
    ''').df()
    assert not frame.duplicated(["client_hash_id", "content_hash_id"]).any(), "Delivered frame is not one row per client-content grain"
    frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
    meta = {"feature_month": feature_month, "feature_window": [str(recent_start), str(cutoff_date)],
            "future_partitions": fut_months, "cutoff": str(cutoff_date), "label_window": [str(label_start), str(label_end)],
            "rows": int(len(frame)), "base_rate": float(frame[TARGET].mean()), "clients": int(frame["client_hash_id"].nunique()),
            "largest_client_share": float(frame["client_hash_id"].value_counts(normalize=True).iloc[0])}
    return frame, meta


def make_baseline_scores(frame):
    # Verbatim re-implementation of the Week-4 hand rule. No fitted parameters.
    is_visible = (frame["recent30_impressions"] >= 500).astype(int)
    is_top10 = ((frame["recent30_avg_position"] > 0) & (frame["recent30_avg_position"] <= 10)).astype(int)
    is_low_ctr = (frame["recent30_ctr_pct"] < 1.0).astype(int)
    is_stale = (frame["content_age_days"] >= 91).astype(int)
    return 0.40 * is_visible + 0.35 * (is_visible * is_top10 * is_low_ctr) + 0.25 * (is_visible * is_stale)


def precision_at_k(labels, scores, k):
    # Copied verbatim from w04_baseline_score.ipynb.
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0


# Week-5 models, unchanged.
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", n_jobs=-1, random_state=SEED,
    ),
}

In [ ]:
# Build both frames, fit on March, score April. This is Week-6 Row 3, reused as the
# operational queue: at the April 30 cutoff the editor gets April's pages ranked by a
# model that only saw March.
df_march, meta_march = build_frame("month=2026-03")
df_april, meta_april = build_frame("month=2026-04")

assert len(df_march) == w6["frames"]["march"]["rows"], f"March frame {len(df_march):,} != W6 receipt"
assert len(df_april) == w6["frames"]["april"]["rows"], f"April frame {len(df_april):,} != W6 receipt"

train = df_march.dropna(subset=FEATURES).copy()
score = df_april.dropna(subset=FEATURES).copy()
print(f"\nTrain (March) rows {len(train):,} | base rate {train[TARGET].mean():.3f}")
print(f"Score (April) rows {len(score):,} | base rate {score[TARGET].mean():.3f} | clients {score['client_hash_id'].nunique()}")

# Rule tie band on March must reproduce the W6 receipt.
train["rule_score"] = make_baseline_scores(train)
tie_march = train["rule_score"] == 1.0
assert int(tie_march.sum()) == TIE["n"], "March tie band does not match the W6 receipt"
print(f"March rule tie band: n={int(tie_march.sum()):,}, decline rate {train.loc[tie_march, TARGET].mean():.3f} (receipt {TIE['decline_rate']:.3f})")

score["rule_score"] = make_baseline_scores(score)
tie_april = score["rule_score"] == 1.0
print(f"April rule tie band: n={int(tie_april.sum()):,}, decline rate {score.loc[tie_april, TARGET].mean():.3f}  <- observed, no receipt yet")

for name, model in models.items():
    model.fit(train[FEATURES], train[TARGET])
    score[f"{name}_proba"] = model.predict_proba(score[FEATURES])[:, 1]

# Momentum: April recent30 impressions against March recent30 impressions for the same page.
# Pages absent from the March frame have no history at the April cutoff.
prev = train[["client_hash_id", "content_hash_id", "recent30_impressions"]].rename(columns={"recent30_impressions": "prev30_impressions"})
score = score.merge(prev, on=["client_hash_id", "content_hash_id"], how="left")
score["impr_ratio"] = score["recent30_impressions"] / score["prev30_impressions"]
score["has_history"] = score["prev30_impressions"].notna()
print(f"April rows with March history: {score['has_history'].mean():.1%}")

tie_april = (score["rule_score"] == 1.0)
april_base = float(score[TARGET].mean())
print("\n=== Time-forward P@50 on the April frame, rebuilt here vs the W6 receipt ===")
check = pd.DataFrame([
    {"scorer": "baseline", "p50_here": precision_at_k(score[TARGET], score["rule_score"], 50), "p50_w6": TF["metrics"]["baseline"]["p50"]},
    {"scorer": "logistic_regression", "p50_here": precision_at_k(score[TARGET], score["logistic_regression_proba"], 50), "p50_w6": TF["metrics"]["logistic_regression"]["p50"]},
    {"scorer": "random_forest", "p50_here": precision_at_k(score[TARGET], score["random_forest_proba"], 50), "p50_w6": TF["metrics"]["random_forest"]["p50"]},
])
print(check.round(3).to_string(index=False))
print(f"April base rate: {april_base:.3f} (receipt {TF['test_base_rate']:.3f})")
print("Baseline P@50 is a draw from inside the tie band, so it can differ from the receipt by tie order alone.")
print("Model P@50 should sit close to the receipt; a large gap means a library version changed the fit (versions printed above).")

In [ ]:
# Reason codes (Week-4 rule flags, verbatim), the four actions, the refresh hint, and the order.
SCORE = "logistic_regression_proba"
BUDGET = 50

_v = (score["recent30_impressions"] >= 500)
_t = (score["recent30_avg_position"] > 0) & (score["recent30_avg_position"] <= 10)
_c = (score["recent30_ctr_pct"] < 1.0)
_s = (score["content_age_days"] >= 91)
_has_stale = _v & _s
_has_lct = _v & _t & _c
score["rule_flag"] = np.select(
    [_has_stale & _has_lct, _has_stale, _has_lct, _v],
    ["stale_low_ctr_top10", "stale", "low_ctr_top10", "visible"],
    default="low_visibility",
)
RULE_ORDER = ["stale_low_ctr_top10", "stale", "low_ctr_top10", "visible", "low_visibility"]
NO_FLAG = {"visible", "low_visibility"}


def action(r):
    ratio = r["impr_ratio"]
    if pd.notna(ratio) and (ratio > 7 or ratio < 1 / 7):
        return "verify_then_review"
    if pd.notna(ratio) and ratio >= 1.5:
        return "monitor_only"
    if r[SCORE] >= 0.70 and r["rule_flag"] in NO_FLAG:
        return "investigate_quiet_risk"
    return "review_before_revert"


def refresh_hint(r):
    if r["recent30_avg_position"] <= 2.0 and r["recent30_ctr_pct"] < 0.20:
        return "diagnose query mix and SERP layout; do not rewrite"
    if r["rule_flag"] in ("stale_low_ctr_top10", "low_ctr_top10"):
        return "title, meta and snippet; then freshness if stale"
    if r["rule_flag"] == "stale":
        return "freshness pass: facts, dates, examples, dead links"
    return "no rule signal; reviewer decides"


def reason_text(r):
    return (f"{r['rule_flag']}; P(decline)={r[SCORE]:.2f}; impr={int(r['recent30_impressions']):,}"
            f"; vs March x{r['impr_ratio']:.2f}" if pd.notna(r["impr_ratio"]) else
            f"{r['rule_flag']}; P(decline)={r[SCORE]:.2f}; impr={int(r['recent30_impressions']):,}; no March history") + (
            f"; pos={r['recent30_avg_position']:.1f}; ctr={r['recent30_ctr_pct']:.2f}%; age={int(r['content_age_days'])}d; active={int(r['recent30_active_days'])}/30")


score["action"] = score.apply(action, axis=1)
score["refresh_hint"] = score.apply(refresh_hint, axis=1)
score["reason_text"] = score.apply(reason_text, axis=1)

ordered = score.sort_values([SCORE, "recent30_impressions"], ascending=[False, False]).reset_index(drop=True)
ordered.insert(0, "rank_all", np.arange(1, len(ordered) + 1))
assert ordered[SCORE].is_monotonic_decreasing

show = ["rank_all", "content_hash_id", "client_hash_id", SCORE, "rule_flag", "action", "recent30_impressions",
        "impr_ratio", "recent30_avg_position", "recent30_ctr_pct", "content_age_days"]
print("=== Top-20 of the April frame before gates (LR fit on March) ===")
print(ordered[show].head(20).round(3).to_string(index=False))

print("\n=== Rule flags on the April frame, with observed decline rate ===")
flag_tbl = pd.concat([
    score["rule_flag"].value_counts().rename("n"),
    score.groupby("rule_flag")[TARGET].mean().round(3).rename("decline_rate"),
    score.groupby("rule_flag")[SCORE].mean().round(3).rename("mean_lr_proba"),
], axis=1).reindex(RULE_ORDER)
print(flag_tbl.to_string())

print("\n=== Actions on the April frame ===")
print(score["action"].value_counts().to_string())

top50_all = ordered.head(BUDGET)
print(f"\nTop-{BUDGET} decline rate before gates (April labels): {top50_all[TARGET].mean():.2f} vs April base {april_base:.3f}")
print(f"Top-{BUDGET} rows in the rule's score-1.00 band: {int((top50_all['rule_flag'] == 'stale_low_ctr_top10').sum())}/{BUDGET}")

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.**

- Who: a content lead or SEO strategist running a monthly refresh sprint across many client sites.
- What: a review order. The queue answers "which of the rule-tied pages should a person look at first this month". It does not answer what to write or whether traffic will recover.
- When: rebuilt on the first working day of each month from the previous month's features, with the model refit on the last labelled month. A queue older than its feature month is expired.
- Budget: 50 pages a month. That is why the metric is precision@50 and why the export is the top 50 eligible rows. Below the budget, neighbouring scores are closer together than the fold-to-fold spread in the Week-5 receipts (P@50 sd about 0.19 to 0.23), so a longer list would be ordering noise.

**Operational thresholds, all stated so they can be argued with.**

- `verify_then_review` at a 7x month-over-month impressions move, either direction.
- `monitor_only` at 1.5x or more, one month. Durability is not measurable from two frames.
- `investigate_quiet_risk` at model score >= 0.70 with no rule flag. 0.70 is the same cut Week 5 used to describe confident picks.
- Gates in Section 3 use 30 days for new pages and 20x for extreme jumps, following the lecture values.
- Below-floor exclusion (100 impressions, 14 covered days) lives upstream in `build_frame`, from the Week-3 contract.

**The decay and refresh finding.** Two things decay here.

1. Content decays with age, but only mildly and not in a straight line. The cell below bins April decline rate by content age. Week 5 measured the same shape in March: the rate peaks around 91-180 days and falls after, which is why LR's linear age term came out negative. Age is useful for choosing the kind of refresh. On its own it is a weak ranker.
2. The ranking itself decays inside weeks. Week 6 measured it: fit on March, RF ranked April's top-50 at 0.14 against a 0.56 base rate and LR at 0.54. On the fixed panel, more history did not fix it, and the base rate moved from 0.20 in February to 0.56 in April. The model is a one-cycle instrument. Refit every month and never act on last month's queue.

**Limits.**

- Observational, not causal. The label is a 20% or larger month-over-month impressions drop. Nothing here shows that a refresh changes that outcome.
- Client- and time-dependent. Grouped-fold P@50 ran from 0.38 to 0.84 across client draws in Week 5, and fell to base rate one month forward in Week 6. Expect it to do best on large stable clients and worst on new or small ones.
- The label partly measures measurement continuity. Week 6 found a 96% decline rate for pages with 14-20 covered days in the outcome month against 40% for 30 days. Some "declines" are tracking gaps.
- Survivorship. 3.2% of March candidates and 3.6% of April candidates were dropped because their outcome-month tracking was thin or absent, and those skew toward pages going quiet. Everything here describes pages measurable next month.
- Clusters. Week 6 found 829 duplicate feature vectors in the March frame and a 182-row single-client cluster behind one perfect fold. The concentration check in Section 3 is a guard, not a fix.
- Position 2 or better with near-zero CTR. The model scores these high; an editor should not act on them. Section 3 gates them for a senior look.

In [ ]:
# Decay finding, part 1: April decline rate by content age (observed, one frame).
age_bins = [0, 30, 90, 180, 365, 730, np.inf]
age_labels = ["<30d", "30-90d", "90-180d", "180-365d", "1-2y", ">2y"]
score["age_bucket"] = pd.cut(score["content_age_days"], bins=age_bins, labels=age_labels, right=False)
age_tbl = (score.groupby("age_bucket", observed=True)
           .agg(pages=(TARGET, "size"), decline_rate=(TARGET, "mean"), median_impr=("recent30_impressions", "median"))
           .reset_index())
age_tbl["pages_pct"] = (100 * age_tbl["pages"] / len(score)).round(1)
print("=== April decline rate by content age (buckets under 500 rows are noisy) ===")
print(age_tbl.round(3).to_string(index=False))

# Decay finding, part 2: P@50 by validation design, all from the W6 receipt.
design_rows = [
    {"design": "grouped CV inside March (mean of 5)", "base": round(w6["frames"]["march"]["base_rate"], 3),
     "rule": GCV["baseline"]["p50_mean"], "LR": GCV["logistic_regression"]["p50_mean"], "RF": GCV["random_forest"]["p50_mean"]},
    {"design": "fit March -> score April", "base": round(TF["test_base_rate"], 3),
     "rule": TF["metrics"]["baseline"]["p50"], "LR": TF["metrics"]["logistic_regression"]["p50"], "RF": TF["metrics"]["random_forest"]["p50"]},
]
for o in w6["walk_forward"]["origins"]:
    design_rows.append({"design": f"walk-forward, history through {o['history_through']} -> April (fixed panel)",
                        "base": round(o["test_base_rate"], 3), "rule": o["metrics"]["baseline"]["p50"],
                        "LR": o["metrics"]["logistic_regression"]["p50"], "RF": o["metrics"]["random_forest"]["p50"]})
design_tbl = pd.DataFrame(design_rows)
print("\n=== P@50 by validation design (W6 receipts) ===")
print(design_tbl.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].bar(age_tbl["age_bucket"].astype(str), age_tbl["decline_rate"], color="#4C72B0")
ax[0].axhline(april_base, ls="--", c="grey", lw=1, label=f"April base rate {april_base:.2f}")
for i, (n, r) in enumerate(zip(age_tbl["pages"], age_tbl["decline_rate"])):
    ax[0].text(i, r + 0.01, f"n={n:,}", ha="center", fontsize=8)
ax[0].set_ylim(0, 1)
ax[0].set_ylabel("observed decline rate (May vs April)")
ax[0].set_title("Content age vs decline rate, April 2026 frame")
ax[0].legend(loc="upper left", fontsize=8)

xs = np.arange(len(design_tbl))
w = 0.25
for j, (m, c) in enumerate([("rule", "#999999"), ("LR", "#DD8452"), ("RF", "#55A868")]):
    ax[1].bar(xs + (j - 1) * w, design_tbl[m], w, label=m, color=c)
ax[1].scatter(xs, design_tbl["base"], marker="_", s=400, c="black", label="test base rate", zorder=3)
ax[1].set_xticks(xs)
ax[1].set_xticklabels(["grouped CV\nMarch", "Mar -> Apr", "wf thru Nov", "wf thru Dec", "wf thru Jan", "wf thru Feb"], fontsize=8)
ax[1].set_ylim(0, 1)
ax[1].set_ylabel("precision@50")
ax[1].set_title("Ranking lift decays out of month (W6 receipts)")
ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
fig1_path = FIG_DIR / "w07_fig1_decay_age_and_model.png"
plt.savefig(fig1_path, dpi=150)
plt.close()
print("\nSaved", fig1_path)

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Review checklist, every queued page, before any edit.** The model sees five numbers. A reviewer checks what it cannot see:

1. Query mix. Open the page's queries in GSC. If impressions are mostly brand or navigational terms, the page is not decaying. Mark it `not_actionable`.
2. SERP layout. Did an AI Overview, featured snippet, video carousel or ad block appear above the result? CTR lost to layout is not a content problem.
3. Technical state. Index status, canonical, redirects, Core Web Vitals, recent template changes. Technical before editorial.
4. Seasonality. Compare with the same window last year where history exists. A seasonal dip needs no action.
5. Business value. Traffic at risk is not value at risk. Does the page matter for conversion, compliance or brand?
6. YMYL or regulated content. Route to the subject-matter owner regardless of score.

The reviewer writes one of `act / defer / not_actionable / escalate` on each row. Those decisions are the labels a later version of this model should learn from. They are worth more than another month of impressions.

**Gates: what is implemented in code and what is proposed.** The cell below applies these before the budget queue is cut. Excluded rows never reach a reviewer; verify rows reach a reviewer with the verify action attached.

| Gate | Rule | Status |
|---|---|---|
| New pages | content age < 30 days, or no March history at the April cutoff | implemented, exclude |
| Durable risers | April impressions >= 1.5x March | implemented as a one-month riser test, exclude; a two-month durability test is proposed, needs a third frame |
| Extreme jumps | April impressions > 20x March or < 1/20 | implemented, verify tracking first |
| Below floor | < 100 impressions or < 14 covered days | implemented upstream in `build_frame` (Week-3 contract) |
| Top-2 with near-zero CTR | position <= 2 and CTR < 0.20% | implemented, senior look first (the Week-5/6 wrong-case pattern) |
| Recently optimised lockout | page changed in the last 30 days | proposed, not implemented; there is no edit-history table in the warehouse release |

**The no-go list. Never automated, whatever the score.**

- No automated deletions, redirects, noindex or consolidation. Irreversible, and the model has no idea why a page exists.
- No unreviewed generative rewrites pushed live. The label cannot tell good content from bad.
- No automated edits to YMYL, legal, medical, financial or safety pages.
- No bulk template or navigation changes triggered by page-level scores. Those are site decisions.
- No client-facing SLAs or reporting built on this score. Fold-to-fold P@50 sd is about 0.2; it is not stable enough to promise a number.
- No acting on `monitor_only` rows and no rewriting top-2 near-zero-CTR rows beyond diagnosis.
- No use of a queue older than its feature month.

In [ ]:
# Gates, applied before the budget queue is cut.
def gates(r):
    ex, ver = [], []
    if r["content_age_days"] < 30 or not r["has_history"]:
        ex.append("NEW_PAGE")
    if pd.notna(r["impr_ratio"]) and r["impr_ratio"] >= 1.5:
        ex.append("RISER_1M")
    if pd.notna(r["impr_ratio"]) and (r["impr_ratio"] > 20 or r["impr_ratio"] < 1 / 20):
        ver.append("EXTREME_JUMP_20X")
    if r["recent30_avg_position"] <= 2.0 and r["recent30_ctr_pct"] < 0.20:
        ver.append("TOP2_NEAR_ZERO_CTR")
    return pd.Series({"gate_exclude": "; ".join(ex), "gate_verify": "; ".join(ver)})


ordered = pd.concat([ordered, ordered.apply(gates, axis=1)], axis=1)
ordered["eligible"] = ordered["gate_exclude"] == ""
GATE_STATUS = {"NEW_PAGE": "implemented, exclude", "RISER_1M": "implemented (one month), exclude; durability proposed",
               "EXTREME_JUMP_20X": "implemented, verify", "TOP2_NEAR_ZERO_CTR": "implemented, senior look first",
               "BELOW_FLOOR": "implemented upstream in build_frame", "RECENTLY_OPTIMISED": "proposed, no data"}

print("=== Gate hits across the April frame ===")
hits = pd.concat([ordered["gate_exclude"], ordered["gate_verify"]]).str.split("; ").explode()
hits = hits[hits != ""].value_counts()
for g, st in GATE_STATUS.items():
    print(f"  {g:<20} {int(hits.get(g, 0)):>7,} rows   [{st}]")

print(f"\nEligible rows: {int(ordered['eligible'].sum()):,} of {len(ordered):,}")
head_excluded = ordered.head(BUDGET)["eligible"].eq(False).sum()
print(f"Rows removed from the ungated top-{BUDGET}: {int(head_excluded)}")

queue = ordered[ordered["eligible"]].copy().reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue["review_decision"] = ""   # act / defer / not_actionable / escalate - filled by a person
queue["reviewer_note"] = ""
top50 = queue.head(BUDGET)

print(f"\n=== Budget queue: top-{BUDGET} eligible rows ===")
print(top50[["rank", "content_hash_id", SCORE, "rule_flag", "action", "gate_verify", "recent30_impressions", "impr_ratio"]].head(20).round(3).to_string(index=False))
print(f"\nTop-{BUDGET} decline rate after gates (April labels): {top50[TARGET].mean():.2f} vs April base {april_base:.3f}")
print("Rule flags in the budget queue:"); print(top50["rule_flag"].value_counts().reindex(RULE_ORDER).fillna(0).astype(int).to_string())
print("Actions in the budget queue:"); print(top50["action"].value_counts().to_string())
cc = top50["client_hash_id"].value_counts(normalize=True)
print(f"\nTop-{BUDGET} spans {cc.size} clients; largest share {cc.iloc[0]:.0%} "
      f"(largest client's share of the April frame: {meta_april['largest_client_share']:.0%})")
if cc.iloc[0] > 0.35:
    print("WARNING: one client holds more than 35% of the queue. Cap per-client rows before handing it to editors.")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a notebook one person runs monthly, not a service. Monitoring is two clocks and one fall-back rule, computed at each rebuild.

**Pre-release clock (before the queue goes out).** Population Stability Index on each of the five features, new frame against the fit frame, plus a schema check that the five features are present with no nulls after the contract filter. PSI bins come from the fit month's deciles. The usual reading is under 0.10 stable, 0.10 to 0.25 watch, over 0.25 act; those are convention, not something measured here. Two more pre-release checks:

- Base-rate drift: the new frame's decline rate moves more than 10 points from the fit month. Week 6 saw 0.20 to 0.56 across February to April.
- Client concentration: the largest client holds more than 35% of the budget queue. The largest client is 22% of the frame.

**Post-release clock (after the outcome month's labels arrive).** Label last month's queue, then compute the true base rate and P@50 for the model and for the rule on the same rows, same K, same tie-break.

**Fall-back policy.** If the model's P@50 is below the rule's P@50 for two consecutive labelled months, rank by the rule until a refit beats it again. A second, stricter check is recorded alongside: whether the model cleared the base rate by 0.05. On the Week-6 evidence LR beat the rule (0.54 vs 0.44) but not the base rate (0.56), so the fall-back does not fire and the stricter check does. Both are printed so the paper can quote either honestly.

**Retrain cadence:** every cycle, unconditionally. There is no model in production to roll back; there is a notebook and a CSV.

**Where this cycle stands.** LR is above the rule out of month and level with a coin flip against the base rate. So this playbook does not claim the queue predicts next month. It claims a review order that beat the rule in every design Week 6 ran, with the rule flag as the reason and the gates removing the known bad rows. If the next labelled month puts the model below the rule again, the fall-back policy applies.

**Cost and value, illustrative.** Editor time is the scarce resource. The cell below assumes rough hours per action (stated in the code, to be replaced with real rates) and sets value at stake to impressions times P(decline). Ranking the budget queue by value per hour shows that the cheapest high-exposure actions are not always the highest-probability rows.

In [ ]:
# Pre-release clock: PSI per feature (March deciles -> April), schema, base-rate drift, concentration.
def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref)
    c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-6, None), np.clip(c, 1e-6, None)
    return float(np.sum((c - r) * np.log(c / r)))


psi_tbl = pd.DataFrame([{"feature": f, "psi_march_to_april": psi(train[f].values, score[f].values)} for f in FEATURES])
psi_tbl["reading"] = pd.cut(psi_tbl["psi_march_to_april"], [-1, 0.10, 0.25, np.inf], labels=["stable", "watch", "act"])
print("=== Pre-release: PSI per feature, March deciles vs April ===")
print(psi_tbl.round(4).to_string(index=False))

checks = []


def add(clock, name, value, threshold, ok, response):
    checks.append({"clock": clock, "trigger": name, "value": value, "threshold": threshold,
                   "status": "PASS" if ok else "TRIGGERED", "response": response})


worst = psi_tbl.loc[psi_tbl["psi_march_to_april"].idxmax()]
add("pre", "feature PSI (worst feature)", f"{worst['feature']} {worst['psi_march_to_april']:.3f}", "< 0.25",
    worst["psi_march_to_april"] < 0.25, "inspect feature; refit before ranking")
schema_ok = all(f in score.columns for f in FEATURES) and score[FEATURES].isna().sum().sum() == 0
add("pre", "schema: five features present, no nulls", "ok" if schema_ok else "broken", "ok", schema_ok, "halt; fix frame query")
drift = abs(april_base - meta_march["base_rate"])
add("pre", "base-rate drift (April vs March)", f"{drift:.3f}", "<= 0.10", drift <= 0.10, "refit before ranking")
add("pre", f"client concentration (top-{BUDGET})", f"{cc.iloc[0]:.0%}", "<= 35%", cc.iloc[0] <= 0.35, "cap per-client rows")

# Post-release clock: last month's queue, labelled. First cycle, so the W6 Mar->Apr receipt stands in.
p_rule, p_lr, p_rf, b = (TF["metrics"]["baseline"]["p50"], TF["metrics"]["logistic_regression"]["p50"],
                         TF["metrics"]["random_forest"]["p50"], TF["test_base_rate"])
add("post", "true base rate of the labelled month", f"{b:.3f}", "recorded", True, "re-read thresholds")
add("post", "fall-back: model P@50 below rule for 2 consecutive months (LR)", f"LR {p_lr:.2f} vs rule {p_rule:.2f}, months below: {int(p_lr < p_rule)}",
    "< 2 months", int(p_lr < p_rule) < 2, "rank by the rule until a refit beats it")
add("post", "fall-back: model P@50 below rule for 2 consecutive months (RF)", f"RF {p_rf:.2f} vs rule {p_rule:.2f}, months below: {int(p_rf < p_rule)}",
    "< 2 months", int(p_rf < p_rule) < 2, "rank by the rule until a refit beats it")
add("post", "stricter: LR P@50 clears base + 0.05", f"{p_lr:.2f} vs {b + 0.05:.2f}", ">= base + 0.05", p_lr >= b + 0.05, "say 'review order', not 'forecast'")

monitor = pd.DataFrame(checks)
print("\n=== Monitoring at this rebuild ===")
print(monitor.to_string(index=False))
triggered = monitor.loc[monitor["status"] == "TRIGGERED", "trigger"].tolist()
print("\nTriggered:", triggered if triggered else "none")

# --- Cost and value (illustrative planning numbers; replace with real rates) ---
HOURS = {"review_before_revert": 3.0, "investigate_quiet_risk": 1.0, "verify_then_review": 0.5, "monitor_only": 0.0}
queue["est_hours"] = queue["action"].map(HOURS)
queue.loc[queue["gate_verify"].str.contains("TOP2_NEAR_ZERO_CTR"), "est_hours"] = 0.25
queue["impr_at_risk"] = queue["recent30_impressions"] * queue[SCORE]
queue["value_per_hour"] = queue["impr_at_risk"] / queue["est_hours"].replace(0, np.nan)
top50 = queue.head(BUDGET)

budget = (top50.groupby("action").agg(rows=("rank", "size"), hours=("est_hours", "sum"), impr_at_risk=("impr_at_risk", "sum")))
budget["impr_at_risk"] = budget["impr_at_risk"].round(0)
print(f"\n=== Effort budget for the top-{BUDGET} by action (illustrative hours) ===")
print(budget.to_string())
print(f"Total: {top50['est_hours'].sum():.0f} editor-hours for {BUDGET} rows.")

print(f"\n=== Same top-{BUDGET} re-ordered by value per editor-hour (top 10) ===")
print(top50.sort_values("value_per_hour", ascending=False)
      [["rank", "rule_flag", "action", SCORE, "recent30_impressions", "est_hours", "value_per_hour"]]
      .head(10).round(2).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
mix = top50["rule_flag"].value_counts().reindex(RULE_ORDER).dropna()
ax[0].barh(mix.index, mix.values, color="#4C72B0")
ax[0].invert_yaxis()
ax[0].set_title(f"Rule flags (reason codes) in the top-{BUDGET} queue")
ax[0].set_xlabel("rows")
cats = top50["rule_flag"].astype("category")
ax[1].scatter(top50[SCORE], top50["recent30_impressions"], c=cats.cat.codes, cmap="tab10", s=28, alpha=0.8)
ax[1].set_yscale("log")
ax[1].set_xlabel("P(decline), LR fit on March")
ax[1].set_ylabel("April 30-day impressions (log)")
ax[1].set_title(f"Top-{BUDGET}: probability vs exposure")
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(i / 10), label=c) for i, c in enumerate(cats.cat.categories)]
ax[1].legend(handles=handles, fontsize=7, loc="lower left")
plt.tight_layout()
fig2_path = FIG_DIR / "w07_fig2_queue_rule_flags.png"
plt.savefig(fig2_path, dpi=150)
plt.close()
print("\nSaved", fig2_path)

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

- `work/outputs/action_playbook_queue.csv`: the top-50 eligible April rows with identifier, model probability (order), Week-4 rule flag (reason), action, refresh hint, gate flags, features, effort estimate and empty reviewer columns. Not committed: the leak-guard ignores CSVs under `work/`, and this notebook regenerates it.
- `work/outputs/action_playbook_summary.json`: committed. Frame sizes, base rates, tie bands, the time-forward check against the Week-6 receipt, rule-flag and action distributions, gate counts with implemented/proposed status, PSI per feature, monitoring statuses, effort assumptions, library versions.
- `work/figures/w07_fig1_decay_age_and_model.png` and `work/figures/w07_fig2_queue_rule_flags.png`: committed, for the recommendations section.

In [ ]:
# Exports.
export_cols = ["rank", "content_hash_id", "client_hash_id", SCORE, "random_forest_proba", "rule_score", "rule_flag",
               "action", "refresh_hint", "gate_verify", "reason_text",
               "recent30_impressions", "prev30_impressions", "impr_ratio", "recent30_avg_position", "recent30_ctr_pct",
               "content_age_days", "recent30_active_days", "est_hours", "impr_at_risk", "value_per_hour",
               "review_decision", "reviewer_note"]
queue_out = queue.head(BUDGET)[export_cols].round(4)
csv_path = OUT_DIR / "action_playbook_queue.csv"
queue_out.to_csv(csv_path, index=False)
print(f"Wrote {len(queue_out)} rows -> {csv_path} (gitignored; rerun this notebook to regenerate)")

summary = {
    "notebook": "w07_action_playbook",
    "seed": SEED,
    "design": "fit Week-5 models on March frame, score April frame (Week-6 Row 3); order by logistic_regression proba, tie-break April impressions; reason = Week-4 rule flag; gates applied before the budget cut",
    "budget": BUDGET,
    "frames": {"train_march": meta_march, "score_april": meta_april},
    "train_rows_after_dropna": int(len(train)),
    "score_rows_after_dropna": int(len(score)),
    "april_rows_with_march_history": float(score["has_history"].mean()),
    "rule_tie_band": {"march": {"n": int(tie_march.sum()), "decline_rate": float(train.loc[tie_march, TARGET].mean())},
                      "april": {"n": int(tie_april.sum()), "decline_rate": float(score.loc[tie_april, TARGET].mean())}},
    "time_forward_p50_check": check.to_dict(orient="records"),
    "april_base_rate": april_base,
    "grouped_cv_p50_receipt": {k: GCV[k]["p50_mean"] for k in GCV},
    "rule_flag_april": flag_tbl.reset_index().rename(columns={"index": "rule_flag"}).to_dict(orient="records"),
    "action_distribution_april": {k: int(v) for k, v in score["action"].value_counts().items()},
    "gates": {"status": GATE_STATUS, "hits_april": {k: int(v) for k, v in hits.items()},
              "eligible_rows": int(ordered["eligible"].sum()), "removed_from_ungated_head": int(head_excluded)},
    "queue": {"size": int(len(queue_out)),
              "top50_decline_rate_before_gates": float(top50_all[TARGET].mean()),
              "top50_decline_rate_after_gates": float(top50[TARGET].mean()),
              "top50_clients": int(cc.size), "top50_largest_client_share": float(cc.iloc[0]),
              "rule_flags": {k: int(v) for k, v in top50["rule_flag"].value_counts().items()},
              "actions": {k: int(v) for k, v in top50["action"].value_counts().items()}},
    "thresholds": {"verify_ratio": 7, "riser_ratio": 1.5, "quiet_risk_proba": 0.70, "new_page_days": 30, "extreme_jump_ratio": 20},
    "age_bucket_decline_rate_april": age_tbl.assign(age_bucket=age_tbl["age_bucket"].astype(str)).to_dict(orient="records"),
    "psi_march_to_april": psi_tbl.assign(reading=psi_tbl["reading"].astype(str)).to_dict(orient="records"),
    "monitoring": monitor.drop(columns=["response"]).to_dict(orient="records"),
    "monitoring_triggered": triggered,
    "effort_hours_assumed": HOURS,
    "top50_total_est_hours": float(top50["est_hours"].sum()),
    "figures": [str(fig1_path.relative_to(REPO)), str(fig2_path.relative_to(REPO))],
    "library_versions": {"pandas": pd.__version__, "numpy": np.__version__,
                         "scikit-learn": sklearn.__version__, "duckdb": duckdb.__version__},
}
json_path = OUT_DIR / "action_playbook_summary.json"
json_path.write_text(json.dumps(summary, indent=2, default=float))
print(f"Receipt written: {json_path}")

# Colab convenience: bundle the committed artefacts.
try:
    from google.colab import files  # noqa: F401
    import shutil
    bundle = Path("/content/w07_artifacts")
    bundle.mkdir(exist_ok=True)
    for p in [json_path, fig1_path, fig2_path]:
        shutil.copy(p, bundle / p.name)
    shutil.make_archive("/content/w07_artifacts", "zip", bundle)
    files.download("/content/w07_artifacts.zip")
    print("Downloaded w07_artifacts.zip: put the JSON in work/outputs/ and the PNGs in work/figures/, then commit.")
except Exception:
    print("Not in Colab (or download skipped): artefacts are already in the repo working tree.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

After the real run, before ticking the boxes:

- The rebuilt time-forward P@50 for LR and RF should sit close to the Week-6 receipt (0.54 and 0.14). If not, check the library versions printed in the first cell before quoting anything.
- If the ungated head of the queue is full of top-2 near-zero-CTR rows, that is the model doing what Week 6 said it would. The gate and the rule flag are what protect the editor. Say so in the paper.
- Read each sentence above on its own. If it claims a result about refresh outcomes, there is no experiment behind it. Rewrite it as decision-support.